# METplus to Parquet

Converts METplus GridStat and EnsembleStat outputs to Parquet format and demonstrates
single-model exploration, cross-model comparison, and ensemble statistics.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

from convert_functions import convert_all

## Settings

Point `INPUT_DIR` at the directory containing your METplus config folders
(named `<model>_<obs>_<parameter>_<timestep>`).  
Output Parquet files will be written to `OUTPUT_DIR`, one per `<model>_<obs>` pair.

In [ ]:
INPUT_DIR  = Path("input_all")   # change this
OUTPUT_DIR = Path("output_all")  # change this

## Convert

Discovers all config folders in `INPUT_DIR`, groups them by model/observation,
and writes one Parquet file per group. Re-running is safe â€” already-loaded
init dates are skipped per parameter and stat type.

In [ ]:
convert_all(INPUT_DIR, OUTPUT_DIR)

## Load datasets

Each Parquet file covers one model vs observation pair and contains all parameters
and timesteps. Use `PARAMETER` and `STAT_TYPE` columns to filter rows.

In [ ]:
g4  = pq.read_table(OUTPUT_DIR / "AGlobal4_Analysis.parquet").to_pandas()
g4e = pq.read_table(OUTPUT_DIR / "AGlobal4E_Analysis.parquet").to_pandas()

print(f"AGlobal4  : {len(g4):,} rows")
print(f"AGlobal4E : {len(g4e):,} rows")

## What's in the data

In [ ]:
print("AGlobal4")
print(g4.groupby(["PARAMETER", "STAT_TYPE"]).size().to_string())
print()
print("AGlobal4E")
print(g4e.groupby(["PARAMETER", "STAT_TYPE"]).size().to_string())

In [ ]:
# Available domains and levels â€” use these to set filters below
print("AGlobal4  domains:", sorted(g4["VX_MASK"].unique()))
print("AGlobal4E domains:", sorted(g4e["VX_MASK"].unique()))
print()
print("AGlobal4  levels (T_AllLevels):", sorted(g4[g4["PARAMETER"] == "T_AllLevels"]["FCST_LEV"].unique()))
print("AGlobal4E levels (T850):",        sorted(g4e[g4e["PARAMETER"] == "T850"]["FCST_LEV"].unique()))

## Column reference

Key columns (â˜…) are the most useful for filtering and plotting.

In [ ]:
COLUMN_INFO = {
    # col name             label                              key?
    "PARAMETER":          ("Parameter (e.g. T_AllLevels)",   True),
    "STAT_TYPE":          ("Stat type (GridStat/EnsembleStat)", True),
    "INIT_DATE":          ("Initialisation Date",             True),
    "FCST_LEAD_H":        ("Forecast Lead Time (hours)",      True),
    "FCST_VALID_BEG":     ("Valid Time",                      True),
    "FCST_LEV":           ("Pressure Level",                  True),
    "VX_MASK":            ("Domain / Region",                 True),
    "TOTAL":              ("Number of Observations",          True),
    # GridStat stats
    "ME":                 ("Mean Error (Bias)",               True),
    "RMSE":               ("Root Mean Square Error",          True),
    "MAE":                ("Mean Absolute Error",             True),
    "FBAR":               ("Mean Forecast Value",             False),
    "OBAR":               ("Mean Observed Value",             False),
    "FOBAR":              ("Mean Forecast * Obs",             False),
    "FFBAR":              ("Mean Forecast Squared",           False),
    "OOBAR":              ("Mean Obs Squared",                False),
    "FABAR":              ("Mean Anomaly Forecast",           False),
    "OABAR":              ("Mean Anomaly Obs",                False),
    "FOABAR":             ("Mean Anomaly Forecast * Obs",     False),
    "FFABAR":             ("Mean Anomaly Forecast Squared",   False),
    "OOABAR":             ("Mean Anomaly Obs Squared",        False),
    # EnsembleStat stats
    "CRPS":               ("Continuous Ranked Probability Score", True),
    "CRPSS":              ("CRPS Skill Score",                True),
    "SPREAD":             ("Ensemble Spread",                 True),
    "N_ENS":              ("Number of Ensemble Members",      False),
    "IGN":                ("Ignorance Score",                 False),
    "ME_OERR":            ("Mean Error (obs error adjusted)", False),
    "RMSE_OERR":          ("RMSE (obs error adjusted)",       False),
    "SPREAD_OERR":        ("Spread (obs error adjusted)",     False),
    "SPREAD_PLUS_OERR":   ("Spread + Obs Error",             False),
    "MAE_OERR":           ("MAE (obs error adjusted)",        False),
    "BIAS_RATIO":         ("Bias Ratio",                      False),
    "CRPS_EMP":           ("CRPS (empirical)",                False),
    "CRPSS_EMP":          ("CRPS Skill Score (empirical)",    False),
}

ref = pd.DataFrame(
    [(col, label, "â˜…" if is_key else "") for col, (label, is_key) in COLUMN_INFO.items() if col in g4.columns or col in g4e.columns],
    columns=["Column", "Description", "Key"],
)
ref.style.apply(
    lambda row: ["font-weight: bold; background-color: #fffbe6"] * 3 if row["Key"] == "â˜…" else [""] * 3,
    axis=1,
)

## Single model â€” GridStat

Explore one parameter from AGlobal4. Change `PARAMETER`, `LEVEL`, and `DOMAIN` to suit.

In [ ]:
PARAMETER = "T_AllLevels"  # change this
LEVEL     = "P850"         # change this
DOMAIN    = "Australia"    # change this
METRIC    = "RMSE"         # change this

subset = g4[
    (g4["PARAMETER"]  == PARAMETER) &
    (g4["STAT_TYPE"]  == "GridStat") &
    (g4["FCST_LEV"]   == LEVEL) &
    (g4["VX_MASK"]    == DOMAIN)
]

print(f"{len(subset):,} rows")
subset.head()

In [ ]:
summary = subset.groupby("FCST_LEAD_H")[METRIC].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(summary.index, summary.values, marker="o")
ax.set_xlabel("Forecast Lead Time (hours)")
ax.set_ylabel(METRIC)
ax.set_title(f"AGlobal4 â€” {PARAMETER} {LEVEL} â€” {METRIC} over {DOMAIN}")
plt.tight_layout()
plt.show()

## Cross-model comparison — GridStat

Compare the same metric for the same parameter across both models.
MSLP is used here as it is present in both datasets at the same level (L0).

**Note:** The AGlobal4E 00Z MSLP data is output as a climatological anomaly
(deviation from the long-term mean) rather than an absolute pressure value,
giving FBAR ~ 0 Pa and RMSE ~ 101,000 Pa for those runs. Set 
to compare only 12Z inits where both models are in absolute units.

In [ ]:
CMP_PARAMETER = "MSLP"   # change this — must exist in both datasets
CMP_LEVEL     = "L0"     # change this
CMP_DOMAIN    = "NH"     # change this — use a domain present in both datasets
CMP_METRIC    = "RMSE"   # change this
CMP_INIT_HOUR = 12       # 0 or 12 — filters to a consistent init time

def _filter(df, parameter, level, domain, init_hour=None):
    mask = (
        (df["PARAMETER"] == parameter) &
        (df["STAT_TYPE"] == "GridStat") &
        (df["FCST_LEV"]  == level) &
        (df["VX_MASK"]   == domain)
    )
    if init_hour is not None:
        mask &= pd.to_datetime(df["INIT_DATE"]).dt.hour == init_hour
    return df[mask]

g4_cmp  = _filter(g4,  CMP_PARAMETER, CMP_LEVEL, CMP_DOMAIN, CMP_INIT_HOUR)
g4e_cmp = _filter(g4e, CMP_PARAMETER, CMP_LEVEL, CMP_DOMAIN, CMP_INIT_HOUR)

print(f"AGlobal4  : {len(g4_cmp):,} rows")
print(f"AGlobal4E : {len(g4e_cmp):,} rows")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for label, df in [("AGlobal4", g4_cmp), ("AGlobal4E", g4e_cmp)]:
    summary = df.groupby("FCST_LEAD_H")[CMP_METRIC].mean()
    ax.plot(summary.index, summary.values, marker="o", label=label)

ax.set_xlabel("Forecast Lead Time (hours)")
ax.set_ylabel(CMP_METRIC)
ax.set_title(f"{CMP_PARAMETER} {CMP_LEVEL} â€” {CMP_METRIC} over {CMP_DOMAIN}")
ax.legend()
plt.tight_layout()
plt.show()

## Cross-model timeseries

Fix a forecast lead time and plot the metric over initialisation dates.
This shows how model performance evolves over time rather than with lead time.
Use a consistent  to avoid mixing absolute and anomaly data.

In [ ]:
TS_PARAMETER = "MSLP"   # change this
TS_LEVEL     = "L0"     # change this
TS_DOMAIN    = "NH"     # change this
TS_METRIC    = "RMSE"   # change this
TS_LEAD_H    = 24       # fixed forecast lead time (hours)
TS_INIT_HOUR = 12       # 0 or 12

def _filter_ts(df, parameter, level, domain, lead_h, init_hour):
    return df[
        (df["PARAMETER"]   == parameter) &
        (df["STAT_TYPE"]   == "GridStat") &
        (df["FCST_LEV"]    == level) &
        (df["VX_MASK"]     == domain) &
        (df["FCST_LEAD_H"] == lead_h) &
        (pd.to_datetime(df["INIT_DATE"]).dt.hour == init_hour)
    ]

g4_ts  = _filter_ts(g4,  TS_PARAMETER, TS_LEVEL, TS_DOMAIN, TS_LEAD_H, TS_INIT_HOUR)
g4e_ts = _filter_ts(g4e, TS_PARAMETER, TS_LEVEL, TS_DOMAIN, TS_LEAD_H, TS_INIT_HOUR)

fig, ax = plt.subplots(figsize=(12, 4))

for label, df in [("AGlobal4", g4_ts), ("AGlobal4E", g4e_ts)]:
    ts = df.set_index("INIT_DATE")[TS_METRIC].sort_index()
    ax.plot(ts.index, ts.values, marker="o", label=label)

ax.set_xlabel("Initialisation Date")
ax.set_ylabel(TS_METRIC)
ax.set_title(
    f"{TS_PARAMETER} {TS_LEVEL} — {TS_METRIC} over {TS_DOMAIN} "
    f"at +{TS_LEAD_H}h ({TS_INIT_HOUR:02d}Z inits)"
)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Ensemble statistics â€” AGlobal4E

EnsembleStat rows contain ensemble-specific metrics: CRPS (lower is better)
and Spread (how much the ensemble members disagree). A well-calibrated ensemble
has Spread â‰ˆ RMSE.

In [ ]:
ENS_PARAMETER = "T850"  # change this â€” must be a parameter with EnsembleStat data
ENS_LEVEL     = "P850"  # change this
ENS_DOMAIN    = "NH"    # change this

ens = g4e[
    (g4e["PARAMETER"] == ENS_PARAMETER) &
    (g4e["STAT_TYPE"] == "EnsembleStat") &
    (g4e["FCST_LEV"]  == ENS_LEVEL) &
    (g4e["VX_MASK"]   == ENS_DOMAIN)
]

print(f"{len(ens):,} rows")
ens[["FCST_LEAD_H", "CRPS", "CRPSS", "SPREAD", "RMSE"]].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# CRPS by lead time
crps = ens.groupby("FCST_LEAD_H")["CRPS"].mean()
axes[0].plot(crps.index, crps.values, marker="o", color="steelblue")
axes[0].set_xlabel("Forecast Lead Time (hours)")
axes[0].set_ylabel("CRPS")
axes[0].set_title(f"AGlobal4E â€” {ENS_PARAMETER} â€” CRPS over {ENS_DOMAIN}")

# Spread vs RMSE by lead time
spread = ens.groupby("FCST_LEAD_H")["SPREAD"].mean()
rmse   = ens.groupby("FCST_LEAD_H")["RMSE"].mean()
axes[1].plot(spread.index, spread.values, marker="o", label="Spread", color="orange")
axes[1].plot(rmse.index,   rmse.values,   marker="o", label="RMSE",   color="steelblue")
axes[1].set_xlabel("Forecast Lead Time (hours)")
axes[1].set_ylabel("Value")
axes[1].set_title(f"AGlobal4E â€” {ENS_PARAMETER} â€” Spread vs RMSE over {ENS_DOMAIN}")
axes[1].legend()

plt.tight_layout()
plt.show()